<a href="https://colab.research.google.com/github/gns1719/Pet-NosePrint-Id-Service/blob/Jun/ResNet%2BSiamses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

In [ ]:
!cat /content/nose_pick/data.yaml

In [ ]:
!pip install PyYAML

In [ ]:
import yaml
dog_nose = {'train' : '/content/nose_pick/train/images/',
        'test' : '/content/nose_pick/test/images',
        'val' : '/content/nose_pick/val/images/',
        'names' : ['nose_data'],
        'nc': 1 }

with open('/content/nose_pick/data.yaml', 'w') as f:
  yaml.dump(dog_nose, f)

with open('/content/nose_pick/data.yaml', 'r') as f:
  display(yaml.safe_load(f))

In [ ]:
!find /content/nose_pick/train -type d -name ".ipynb_checkpoints" -exec rm -rf {} +
!find /content/nose_pick/val -type d -name ".ipynb_checkpoints" -exec rm -rf {} +&#8203;:contentReference[oaicite:5]{index=5}

In [ ]:
# STEP 1: 필수 라이브러리 설치
!pip install -q torch torchvision PyYAML

# STEP 2: 필요한 라이브러리 불러오기
import os
import yaml
import torch
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch import nn, optim

# STEP 3: YAML 파일 로드
with open('/content/nose_pick/data.yaml', 'r') as f:
    data_config = yaml.safe_load(f)

train_dir = data_config['train']
val_dir = data_config['val']
num_classes = data_config['nc']
class_names = data_config['names']

# STEP 4: 데이터셋 준비
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# STEP 5: ResNet 모델 로드 및 수정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(pretrained=True)  # ResNet18 사용
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# STEP 6: 손실 함수 및 옵티마이저 설정
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# STEP 7: 학습 루프
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100. * correct / total
    print(f"[Epoch {epoch+1}] Loss: {running_loss:.4f}, Train Acc: {train_acc:.2f}%")

    # 검증
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    val_acc = 100. * correct / total
    print(f"Validation Accuracy: {val_acc:.2f}%")

# STEP 8: 모델 저장
torch.save(model.state_dict(), "/content/resnet_nose_pick.pth")

In [ ]:
import torch
from torchvision import models

# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 모델 로드
model = models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(torch.load("/content/resnet_nose_pick.pth", map_location=device))
model = model.to(device)
model.eval()

# 마지막 분류 계층 제거
feature_extractor = torch.nn.Sequential(*list(model.children())[:-1])  # avgpool까지 포함
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

In [ ]:
import torch
from torchvision import models, transforms
from PIL import Image

# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 모델 로드 및 수정
model = models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)  # num_classes는 학습 시 사용한 클래스 수로 설정
model.load_state_dict(torch.load("/content/resnet_nose_pick.pth", map_location=device))
model = model.to(device)
model.eval()

# 특징 추출기 설정 (분류 계층 제거)
feature_extractor = torch.nn.Sequential(*list(model.children())[:-1])  # avgpool까지 포함
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

# 이미지 전처리
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 이미지 로드 및 전처리
img_path = "/content/photo/test/coco2.jpg"  # 예시 이미지 경로
image = Image.open(img_path).convert("RGB")
input_tensor = transform(image).unsqueeze(0).to(device)

# 특징 벡터 추출 및 저장
with torch.no_grad():
    features = feature_extractor(input_tensor)  # 출력 크기: [1, 512, 1, 1]
    embedding = features.view(features.size(0), -1)  # [1, 512]
    torch.save(embedding, "/content/photo/result/coco2.pt")
    print("임베딩 벡터가 /content/embedding_1.pt에 저장되었습니다.")

In [ ]:
import torch
import torch.nn.functional as F
import os

# ===== 1. 경로 설정 =====
target_path = "/content/photo/result/target.pt"
db_folder_path = "/content/photo/result/db_vectors"

# ===== 2. 유사도 및 거리 계산 함수 =====
def cosine_similarities(target_embedding, db_embeddings):
    target = target_embedding.unsqueeze(0).expand(db_embeddings.size(0), -1)
    return F.cosine_similarity(target, db_embeddings)

def euclidean_distances(target_embedding, db_embeddings):
    target = target_embedding.unsqueeze(0).expand(db_embeddings.size(0), -1)
    return F.pairwise_distance(target, db_embeddings)

# ===== 3. 임베딩 로드 =====
# 비교 대상 벡터
target_embedding = torch.load(target_path)  # (D,)

# DB 벡터들 불러오기
embedding_files = sorted([f for f in os.listdir(db_folder_path) if f.endswith('.pt')])
db_embeddings = []
file_names = []  # 나중에 어떤 파일과 유사했는지 확인용
for file in embedding_files:
    emb = torch.load(os.path.join(db_folder_path, file))
    db_embeddings.append(emb)
    file_names.append(file)

# 텐서로 묶기 (N, D)
db_embeddings_tensor = torch.stack(db_embeddings)

# ===== 4. 유사도 계산 =====
cos_similarities = cosine_similarities(target_embedding, db_embeddings_tensor)
eucl_distances = euclidean_distances(target_embedding, db_embeddings_tensor)

# ===== 5. 결과 출력 =====
threshold_cosine = 0.8
threshold_euclidean = 1.0

print("🔍 이미지 임베딩 비교 결과:")
for i, (cos_sim, euc_dist) in enumerate(zip(cos_similarities, eucl_distances)):
    print(f"\n[{i+1}] {file_names[i]}")
    print(f"코사인 유사도: {cos_sim:.4f}")
    print(f"유클리드 거리: {euc_dist:.4f}")

    if cos_sim > threshold_cosine:
        print("→ ✅ 코사인 유사도 기준: 유사합니다.")
    else:
        print("→ ❌ 코사인 유사도 기준: 다릅니다.")

    if euc_dist < threshold_euclidean:
        print("→ ✅ 유클리드 거리 기준: 유사합니다.")
    else:
        print("→ ❌ 유클리드 거리 기준: 다릅니다.")
